## Supply chain and logistics management agent

``Supply_chain_logistics_agent.py``  

from  book repo
LangGraph workflow for a supply chain and logistics management agent,  
handling inventory management, shipping operations, supplier relations and warehouse optimization.

#### define tools for single agent

In [ ]:
# import section
from langchain.tools import tool
from src.loki_logger import log_to_loki

#should also run loki
#docker run -d --name loki -p 3100:3100 grafana/loki:latest \
#   -config.file=/etc/loki/local-config.yaml
#and grafana:
# docker run -d \
#   --name grafana \
#   -p 3000:3000 \
#   grafana/grafana:latest

In [21]:
@tool
def manage_inventory(sku: str=None, **kwargs) -> str:
    """Manage inventory levels, stock replenishments, audits, and optimization strategies."""
    print(f"[TOOL] manage_inventory(sku={sku}, kwargs={kwargs})")
    log_to_loki("tool.manage_inventory", f"sku={sku}")
    return "inventory_management_initiated"

@tool
def track_shipments(origin: str=None, **kwargs) -> str:
    """Track shipment status, delays, and cordinate delivery logistics."""
    print(f"[TOOL] track_shipments(origin={origin}, kwargs={kwargs})")
    log_to_loki("tool.track_shipments", f"origin={origin}")
    return "shipment_tracking_updated"

@tool
def evaluate_suppliers(supplier_name: str=None, **kwargs) -> str:
    """Evaluate supllier performance, conduct audits, and manage supplier relationships."""
    print(f"[TOOL] evaluate_suppliers(supplier_name={supplier_name}, kwargs={kwargs})")
    log_to_loki("tool.evaluate_suppliers", f"supplier_name={supplier_name}")
    return "supplier_evaluation_complete"

@tool
def optimize_warehouse(operation_type: str=None, **kwargs) -> str:
    """Optimize warehouse operations, layout, capacity, and storage efficiency."""
    print(f"[TOOL] optimize_warehouse(operation_type={operation_type}, kwargs={kwargs})")
    log_to_loki("tool.optimize_warehouse", f"operation_type={operation_type}")
    return "warehouse_optimization_initiated"

@tool
def forecast_demand(season: str=None, **kwargs) -> str:
    """Analyze demand patterns, seasonal trends, and create forecasting models."""
    print(f"[TOOL] forcast_demand(season={season}, kwargs={kwargs})")
    log_to_loki("tool.forcast_demand", f"season={season}")
    return "demand_forecast_generated"

@tool
def manage_quality(supplier: str = None, **kwargs) -> str:
    """Manage quality control, defect tracking, and supplier quality standards."""
    print(f"[TOOL] manage_quality(supplier={supplier}, kwargs={kwargs})")
    log_to_loki("tool.manage_quality", f"supplier={supplier}")
    return "quality_management_initiated"
@tool
def arrange_shipping(shipping_type: str = None, **kwargs) -> str:
    """Arrange shipping methods, expedited delivery, and multi-modal transportation."""
    print(f"[TOOL] arrange_shipping(shipping_type={shipping_type}, kwargs={kwargs})")
    log_to_loki("tool.arrange_shipping", f"shipping_type={shipping_type}")
    return "shipping_arranged"

@tool
def coordinate_operations(operation_type: str = None, **kwargs) -> str:
    """Coordinate complex operations like cross-docking, consolidation, and transfers."""
    print(f"[TOOL] coordinate_operations(operation_type={operation_type}, kwargs={kwargs})")
    log_to_loki("tool.coordinate_operations", f"operation_type={operation_type}")
    return "operations_coordinated"

@tool
def manage_special_handling(product_type: str = None, **kwargs) -> str:
    """Handle special requirements for hazmat, cold chain, and sensitive products."""
    print(f"[TOOL] manage_special_handling(product_type={product_type}, kwargs={kwargs})")
    log_to_loki("tool.manage_special_handling", f"product_type={product_type}")
    return "special_handling_managed"

@tool
def handle_compliance(compliance_type: str = None, **kwargs) -> str:
    """Manage regulatory compliance, customs, documentation, and certifications."""
    print(f"[TOOL] handle_compliance(compliance_type={compliance_type}, kwargs={kwargs})")
    log_to_loki("tool.handle_compliance", f"compliance_type={compliance_type}")
    return "compliance_handled"

@tool
def process_returns(returned_quantity: str = None, **kwargs) -> str:
    """Process returns, reverse logistics, and product disposition."""
    print(f"[TOOL] process_returns(returned_quantity={returned_quantity}, kwargs={kwargs})")
    log_to_loki("tool.process_returns", f"returned_quantity={returned_quantity}")
    return "returns_processed"

@tool
def scale_operations(scaling_type: str = None, **kwargs) -> str:
    """Scale operations for peak seasons, capacity planning, and workforce management."""
    print(f"[TOOL] scale_operations(scaling_type={scaling_type}, kwargs={kwargs})")
    log_to_loki("tool.scale_operations", f"scaling_type={scaling_type}")
    return "operations_scaled"

@tool
def optimize_costs(cost_type: str = None, **kwargs) -> str:
    """Analyze and optimize transportation, storage, and operational costs."""
    print(f"[TOOL] optimize_costs(cost_type={cost_type}, kwargs={kwargs})")
    log_to_loki("tool.optimize_costs", f"cost_type={cost_type}")
    return "cost_optimization_initiated"

@tool
def optimize_delivery(delivery_type: str = None, **kwargs) -> str:
    """Optimize delivery routes, last-mile logistics, and sustainability initiatives."""
    print(f"[TOOL] optimize_delivery(delivery_type={delivery_type}, kwargs={kwargs})")
    log_to_loki("tool.optimize_delivery", f"delivery_type={delivery_type}")
    return "delivery_optimization_complete"

@tool
def manage_disruption(disruption_type: str = None, **kwargs) -> str:
    """Manage supply chain disruptions, contingency planning, and risk mitigation."""
    print(f"[TOOL] manage_disruption(disruption_type={disruption_type}, kwargs={kwargs})")
    log_to_loki("tool.manage_disruption", f"disruption_type={disruption_type}")
    return "disruption_managed"

@tool
def send_logistics_response(operation_id: str = None, message: str = None) -> str:
    """Send logistics updates, recommendations, or status reports to stakeholders."""
    print(f"[TOOL] send_logistics_response → {message}")
    log_to_loki("tool.send_logistics_response", f"operation_id={operation_id}, message={message}")
    return "logistics_response_sent"

TOOLS = [
    manage_inventory, track_shipments, evaluate_suppliers, optimize_warehouse,
    forecast_demand, manage_quality, arrange_shipping, coordinate_operations,
    manage_special_handling, handle_compliance, process_returns, scale_operations,
    optimize_costs, optimize_delivery, manage_disruption, send_logistics_response
]


#### Agent setup  
- foundation model, model binding, state definition, and graph construction

In [31]:
from traceloop.sdk import Traceloop
from langchain_ollama import ChatOllama
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from typing import TypedDict, Optional, Annotated
from langchain_core.messages import BaseMessage,SystemMessage, ToolMessage,HumanMessage
import json
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages


Traceloop.init(disable_batch=True, app_name="supply_chain_logistics_agent", api_key="tl_d32b11245b734b5386e82487afec0d46")
llm_model = ChatOllama(model="llama3.1:latest",
                        temperature=0.0,
                        callbacks=[StreamingStdOutCallbackHandler()],
                        verbose=True).bind_tools(TOOLS)

class AgentState(TypedDict):
    operation: Optional[dict]
    messages: Annotated[list[BaseMessage], add_messages]

def call_model(state: AgentState):
    history = state.get("messages", [])
    
    #handle missing or incomplete operation data gracefully
    operation = state.get("operation", {})
    if not operation:
        operation = {"operation_id": "UNKNOWN", "type": "general",
                     "priority": "medium", "status": "active"}
    operation_json = json.dumps(operation, ensure_ascii=False)
    system_prompt = (
        "You are an experienced Supply Chain & Logistics Management professional.\n"
        "Your expertise covers:\n"
        "- Inventory management and demand forecasting\n"
        "- Transportation and shipping optimization\n"
        "- Supplier relationship management and evaluation\n"
        "- Warehouse operations and capacity planning\n"
        "- Quality control and compliance management\n"
        "- Cost optimization and operational efficiency\n"
        "- Risk management and disruption response\n"
        "- Sustainability and green logistics initiatives\n"
        "\n"
        "When managing supply chain operations:\n"
        "  1) Analyze the logistics challenge or opportunity\n"
        "  2) Call the appropriate supply chain management tool\n"
        "  3) Follow up with send_logistics_response to provide recommendations\n"
        "  4) Consider cost, efficiency, quality, and sustainability impacts\n"
        "  5) Prioritize customer satisfaction and business continuity\n"
        "\n"
        "Always balance cost optimization with service quality and risk mitigation.\n\n"
        f"OPERATION: {operation_json}"
    )

    full = [SystemMessage(content=system_prompt)] + history

    first: ToolMessage | BaseMessage = llm_model.invoke(full)
    messages = [first]

    if getattr(first, "tool_calls", None):
        for tc in first.tool_calls:
            print(first)
            print(tc['name'])
            fn = next(t for t in TOOLS if t.name == tc['name'])
            out = fn.invoke(tc["args"])
            messages.append(ToolMessage(content=str(out), tool_call_id=tc["id"]))

        second = llm_model.invoke(full + messages)
        messages.append(second)

    return {"messages": messages}

def construct_graph():
    g = StateGraph(AgentState)
    g.add_node("assistant", call_model)
    g.set_entry_point("assistant")
    return g.compile()

graph = construct_graph()

if __name__ == "__main__":
    example = {"operation_id": "OP-12345", "type": "inventory_management", "priority": "high", "location": "Warehouse A"}
    convo = [HumanMessage(content="We're running critically low on SKU-12345. Current stock is 50 units but we have 200 units on backorder. What's our reorder strategy?")]
    result = graph.invoke({"operation": example, "messages": convo})
    for m in result["messages"]:
        print(f"{m.type}: {m.content}") 


Traceloop exporting traces to https://api.traceloop.com authenticating with bearer token

content='' additional_kwargs={} response_metadata={'model': 'llama3.1:latest', 'created_at': '2026-04-09T12:53:43.693466692Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3866375562, 'load_duration': 2996354291, 'prompt_eval_count': 1065, 'prompt_eval_duration': 293402939, 'eval_count': 39, 'eval_duration': 546530960, 'logprobs': None, 'model_name': 'llama3.1:latest', 'model_provider': 'ollama'} id='lc_run--019d724e-2f72-7680-b855-1e1b18d44b70-0' tool_calls=[{'name': 'manage_inventory', 'args': {'kwargs': {'reorder_point': 100, 'lead_time': 7}, 'sku': 'SKU-12345'}, 'id': '3ef94aea-cf65-49f7-b3ee-690e775b835a', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 1065, 'output_tokens': 39, 'total_tokens': 1104}
manage_inventory
[TOOL] manage_inventory(sku=SKU-12345, kwargs={'kwargs': {'reorder_point': 100, 'lead_time': 7}})
Status: 204
Response: 
Based on the inven